<h1>Chapter 1 - RAG Setup</h1>
<i>Building your first RAG setup.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch01_RAG_intro/rag_basics.ipynb)

---

This notebook is for Chapter 1 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


## Prerequisits

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to uncomment and run the following codeblock to install the dependencies for this chapter.

In [5]:
!pip install openai
!pip install chromadb

### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [30]:
from google.colab import userdata
import os

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in Colab Secrets")

os.environ["OPENAI_API_KEY"] = api_key

### Load sample files

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [24]:
import os
from pathlib import Path

repo_name = "RAG-with-Python-Cookbook"
repo_path = Path('/content') / repo_name

# Ensure a clean slate for cloning
if repo_path.exists():
    !rm -rf {repo_path}

# Clone the repository without checking out all files initially
!git clone --no-checkout https://github.com/polzerdo55862/{repo_name}.git {repo_path}

# Change directory to the cloned repository to perform sparse checkout
# This ensures sparse-checkout commands are run from the correct location
%cd {repo_path}

# Initialize sparse checkout and set the specific file path
!git sparse-checkout init --cone
!git sparse-checkout set datasets/chapter1/harry_potter.txt
!git checkout

# Change back to the /content directory for consistent absolute pathing in subsequent cells
%cd /content/

print(f"Current working directory: {os.getcwd()}")
print(f"harry_potter.txt should now be at: {repo_path / 'datasets' / 'chapter1' / 'harry_potter.txt'}")

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Cloning into '/content/RAG-with-Python-Cookbook'...
fatal: Unable to read current working directory: No such file or directory
[Errno 2] No such file or directory: '/content/RAG-with-Python-Cookbook'
/content/RAG-with-Python-Cookbook/RAG-with-Python-Cookbook/RAG-with-Python-Cookbook/RAG-with-Python-Cookbook
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
/c

## 1. Chunking Text

In [7]:
import chromadb
import openai

In [33]:
def chunk_text(text, chunk_size, overlap):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size

        if end < len(text):
            break_point = text.rfind("\n\n", start, end)
            if break_point == -1:
                break_point = text.rfind(". ", start, end)
            if break_point != -1 and break_point > start:
                end = break_point + 1

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        start = end - overlap if end < len(text) else end

    return chunks

knowledge_base_file = "harry_potter.txt"  # Path to your knowledge base file
chunk_size = 1000  # Number of characters per chunk
chunk_overlap = 200  # Number of overlapping characters between chunks

# Explicitly set the file path to the expected location after cloning and sparse checkout
file_path = Path('/content/RAG-with-Python-Cookbook/datasets') / 'chapter1' / knowledge_base_file

if not file_path.exists():
    print(f"Error: Knowledge base file not found at {file_path}")
    # Fallback if path is incorrect for some reason (shouldn't be needed after OTdB78be8XVn runs correctly)
    # If this message is printed, inspect the directory structure.

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

chunks = chunk_text(text, chunk_size, chunk_overlap)

Error: Knowledge base file not found at /content/RAG-with-Python-Cookbook/datasets/chapter1/harry_potter.txt


FileNotFoundError: [Errno 2] No such file or directory: '/content/RAG-with-Python-Cookbook/datasets/chapter1/harry_potter.txt'

In [34]:
def generate_embeddings(texts, client, model):
    embeddings = []
    batch_size = 100

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        response = client.embeddings.create(model=model, input=batch)
        embeddings.extend([item.embedding for item in response.data])

    return embeddings

client = OpenAI()
embedding_model = "text-embedding-ada-002"

embeddings = generate_embeddings(chunks, client, embedding_model)

In [41]:
def ingest_to_chromadb(chunks, embeddings, db_path, collection_name):

    db_path.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(db_path))

    try:
        client.delete_collection(name=collection_name)
    except:
        pass

    collection = client.create_collection(
        name=collection_name, metadata={"description": "Harry Potter knowledge base"}
    )

    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        embeddings=embeddings,
        documents=chunks,
        metadatas=[{"chunk_index": i} for i in range(len(chunks))],
    )

    return collection.count()

chroma_db_dir = Path("chroma_db")  # Directory for ChromaDB persistence
collection_name = "harry_potter_kb"  # Name of the ChromaDB collection

count = ingest_to_chromadb(chunks, embeddings, chroma_db_dir, collection_name)
count

28

In [40]:
# from pathlib import Path
# from openai import OpenAI
# import os


# def main():
#     knowledge_base_file = "harry_potter.txt"  # Name of the knowledge base file
#     chunk_size = 1000  # Number of characters per chunk
#     chunk_overlap = 200  # Number of overlapping characters between chunks
#     embedding_model = "text-embedding-ada-002"  # OpenAI embedding model name
#     chroma_db_dir = Path("chroma_db")  # Directory for ChromaDB persistence
#     collection_name = "harry_potter_kb"  # Name of the ChromaDB collection

#     # Explicitly set the path to the knowledge base file
#     # The datasets directory is expected to be inside the cloned repo after sparse checkout
#     file_path = Path('/content/RAG-with-Python-Cookbook/datasets') / 'chapter1' / knowledge_base_file

#     if not file_path.exists():
#         print(f"Error: Knowledge base file not found at {file_path}")
#         return

#     with open(file_path, "r", encoding="utf-8") as f:
#         text = f.read()

#     chunks = chunk_text(text, chunk_size, chunk_overlap)

#     client = OpenAI()
#     embeddings = generate_embeddings(chunks, client, embedding_model)

#     count = ingest_to_chromadb(chunks, embeddings, chroma_db_dir, collection_name)
#     print(f"Successfully ingested {count} chunks into ChromaDB collection '{collection_name}'")


# if __name__ == "__main__":
#     main()